In [67]:
import polars as pl
import pandas as pd
import numpy as mp
import altair as alt
import pyarrow
from numpy.ma.extras import unique
from polars import Null

from func.UsefulFunctions import *

This notebook documents the cleaning and inspection process for the artist data on wikidata

In [68]:
artistDF = pl.read_parquet("./Data/pipelinesArtistsWikidata.parquet")
print(f"There are {len(artistDF["artist"].unique())} unique artist entities after the removal of the disallowed occupations.")
display(artistDF.head(3))

There are 444888 unique artist entities after the removal of the disallowed occupations.


artist,occupation,gender,occupationLabel,genderLabel,countryCitizenship,influencedBY,influencedBYLabel,statementCount,genre,article,musicBrainzID,recordLabel,recordLabelLabel,countryCitizenshipLabel,genreLabel,artistLabel
str,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,str
"""Q7161""","""Q36834""","""Q6581097""","""composer""","""male""","""Q29""",null,null,null,"""Q1413570""","""https://en.wikipedia.org/wiki/…","""533761c0-61cd-4d0e-b264-c71d5d…",null,null,"""Spain""","""sardana""","""José María Ventura Casas"""
"""Q7241""","""Q36834""","""Q6581097""","""composer""","""male""","""Q129286""",null,null,538.0,null,"""https://en.wikipedia.org/wiki/…","""13274b60-181b-431a-b049-b48560…",null,null,"""British Raj""",null,"""Rabindranath Tagore"""
"""Q7241""","""Q36834""","""Q6581097""","""composer""","""male""","""Q129286""",null,null,538.0,null,"""https://en.wikipedia.org/wiki/…","""13274b60-181b-431a-b049-b48560…",null,null,"""British Raj""",null,"""Rabindranath Tagore"""


In [69]:
#inspect the different fields in the system that are error prone
#see the different genders in the dataframe
print(artistDF[["artist","genderLabel"]].unique().select(pl.col("genderLabel").value_counts(sort=True)).unnest("genderLabel"))
print(artistDF[["genderLabel"]].unique().select(pl.col("genderLabel")).to_series().to_list())

#see some genders who have unknown or none gender
noGender = artistDF.filter(
    pl.col("gender").is_null()
)
noGender = noGender.select(pl.col(["artist","gender","genderLabel"])).unique()
print(noGender)
#Save the file for individuals who do not have a gender
noGender.write_csv("./missingValuesStore/artistsEntitiesWithoutGenders.csv",)


shape: (30, 2)
┌──────────────────┬────────┐
│ genderLabel      ┆ count  │
│ ---              ┆ ---    │
│ str              ┆ u32    │
╞══════════════════╪════════╡
│ male             ┆ 331681 │
│ female           ┆ 112112 │
│ non-binary       ┆ 472    │
│ trans woman      ┆ 471    │
│ trans man        ┆ 107    │
│ …                ┆ …      │
│ gender agnostic  ┆ 1      │
│ apagender        ┆ 1      │
│ non-binary woman ┆ 1      │
│ neutral sex      ┆ 1      │
│ demiboy          ┆ 1      │
└──────────────────┴────────┘
['apagender', 'demigirl', 'trans woman', 'trans man', 'undisclosed gender', 'male', 'genderqueer', 'cisgender woman', 'transfeminine', 'genderfluid', 'two-spirit', 'transgender', 'agender', 'non-binary', 'travesti', 'androgyne', 'female', 'intersex woman', 'gender agnostic', 'intersex', 'cisgender man', 'demiboy', 'takatāpui', 'intersex man', 'bigender', 'intersex trans woman', None, 'neutral sex', 'non-binary woman', 'transmasculine']
shape: (41, 3)
┌────────────┬──────

Checked the genders to ensure that error/unknown genders were successfully nulled.

In [70]:
#Null the artist labels for the individuals who do not have an artist label

noNames = artistDF.filter(
    pl.col("artist").is_in(pl.col("artistLabel").implode())
).select(pl.col("artistLabel")).unique()
name_dict = {k: None for k in noNames["artistLabel"].to_list()}
artistDF = artistDF.with_columns(pl.col("artistLabel").replace(name_dict))

noNamesDF = artistDF.filter(
    pl.col("artistLabel").is_null(),
).select(pl.col(["artist","artistLabel"])).unique()
display(noNamesDF)
print(f"There are {len(noNamesDF)} artists with no default labels/labels in english.")
noNamesDF.write_csv("./missingValuesStore/artistsEntitiesWithoutLabels.csv")

artist,artistLabel
str,str
"""Q20607442""",null
"""Q48644710""",null
"""Q12103442""",null
"""Q13204391""",null
"""Q26840620""",null
…,…
"""Q112976972""",null
"""Q137185613""",null
"""Q134276406""",null


There are 14131 artists with no default labels/labels in english.


In [71]:
#The individuals with a country without an english label
#First let's see if there are any unknown or erroneous countries by using the Q-numbers and there are
artistDF.filter(
    pl.col('countryCitizenshipLabel').str.starts_with("Q")
).select(pl.col('countryCitizenshipLabel')).unique().to_series().to_list()

noLabel_Country = artistDF.filter(
    pl.col("countryCitizenship").is_in(pl.col("countryCitizenshipLabel").implode())
).select(pl.col("countryCitizenshipLabel")).unique()
countryName_dict = {k: None for k in noLabel_Country["countryCitizenshipLabel"].to_list()}
artistDF = artistDF.with_columns(pl.col("countryCitizenshipLabel").replace(countryName_dict))
artistDF = artistDF.with_columns(
    countryCitizenship = pl.when(~pl.col('countryCitizenship').str.starts_with("Q")).then(None).otherwise(pl.col('countryCitizenship'))
)
noCountryLabel = artistDF.filter(
    pl.col("countryCitizenshipLabel").is_null(),
).select(pl.col(["countryCitizenship","countryCitizenshipLabel"])).unique()
display(noCountryLabel)
# noCountry = artistDF.filter(
#     pl.col("countryCitizenship").is_in(pl.col("countryCitizenshipLabel").implode())
# ).select(pl.col("countryCitizenship")).unique()
# display(noCountry)
# print(f"There are {len(noCountry)} artists with a country with no labels. These will be cleaned out.")
# noCountry.write_csv("./missingValuesStore/artistsEntitiesWithoutLabelledCountries.csv")

countryCitizenship,countryCitizenshipLabel
str,str
"""Q20581158""",null
"""Q2181059""",null
null,null


In [72]:
#Genres with no labels in english/default language
noGenre = artistDF.filter(
    pl.col("genre").is_in(pl.col("genreLabel").implode())
).select(pl.col('genreLabel')).unique()
noGenreName_dict = {k: None for k in noGenre["genreLabel"].to_list()}
artistDF = artistDF.with_columns(pl.col("genreLabel").replace(noGenreName_dict))
artistDF = artistDF.with_columns(
    genre = pl.when(~pl.col('genre').str.starts_with("Q")).then(None).otherwise(pl.col('genre'))
)

noGenreLabel = artistDF.filter(
    pl.col("genreLabel").is_null(),
).select(pl.col(["artist","artistLabel","genre","genreLabel"])).unique()
display(noGenreLabel)
#print(f"There are {len(noGenre)} artists with a genre with no labels. These will be cleaned out.")
noGenreLabel.write_csv("./missingValuesStore/artistsEntitiesWithoutLabelledGenresOrNoGenres.csv")

artist,artistLabel,genre,genreLabel
str,str,str,str
"""Q12728960""","""George Cosmovici""",null,null
"""Q105925041""","""Kent Finlay""",null,null
"""Q85681729""","""Rachid Batma""",null,null
"""Q82283321""","""Michael Sheridan""",null,null
"""Q117218443""","""Maybelle Boma""",null,null
…,…,…,…
"""Q1521315""","""Ghadah al-Akel""",null,null
"""Q131328658""","""José Balaguer""",null,null
"""Q18507907""","""Franja Golob""",null,null


In [73]:
#Record Labels without a label
noRecordLabelLabel = artistDF.filter(
    pl.col("recordLabel").is_in(pl.col("recordLabelLabel").implode())
).select(pl.col(["recordLabelLabel"])).unique()
recordLabel_dict = {k: None for k in noRecordLabelLabel["recordLabelLabel"].to_list()}
artistDF = artistDF.with_columns(
    pl.col("recordLabelLabel").replace(recordLabel_dict),
    recordLabel = pl.when(~pl.col('recordLabel').str.starts_with("Q")).then(None).otherwise(pl.col('recordLabel')),
)

recordLabelsWithNoLabel = artistDF.filter(
    pl.col("recordLabelLabel").is_null()
).select(["artist","artistLabel","recordLabel","recordLabelLabel"]).unique()
display(recordLabelsWithNoLabel)

recordLabelsWithNoLabel.write_csv("./missingValuesStore/artistsEntitiesWithoutLabelledRecordLabelsOrNoRecordLabels.csv")

artist,artistLabel,recordLabel,recordLabelLabel
str,str,str,str
"""Q16232809""","""Magnus Nergaard""",null,null
"""Q503491""","""August von Adelburg""",null,null
"""Q12369671""","""Marek Talivere""",null,null
"""Q9281818""","""Grzegorz Grzyb""",null,null
"""Q108894103""","""Simeon Pironkoff""",null,null
…,…,…,…
"""Q12787288""","""Danijel Černe""",null,null
"""Q95464232""","""Petr Horák""",null,null
"""Q107602696""","""Robert Wienröder""",null,null


In [74]:
#Influences Labels without a label
noLabelsInfluencedBY = artistDF.filter(
    pl.col("influencedBY").is_in(pl.col("influencedBYLabel").implode())
).select(pl.col(["influencedBYLabel"])).unique()

influencedBName_dict = {k: None for k in noLabelsInfluencedBY["influencedBYLabel"].to_list()}
artistDF = artistDF.with_columns(
    pl.col("influencedBYLabel").replace(influencedBName_dict),
    influencedBY = pl.when(~pl.col('influencedBY').str.starts_with("Q")).then(None).otherwise(pl.col('influencedBY'))
)

entitiesWithNullInfluences = artistDF.filter(
    pl.col("influencedBYLabel").is_null()
).select(pl.col(["artist","artistLabel","influencedBY","influencedBYLabel"])).unique()
display(entitiesWithNullInfluences)
entitiesWithNullInfluences.write_csv("./missingValuesStore/artistsEntitiesWithoutLabelledInfluencesOrNoInfluences.csv")

artist,artistLabel,influencedBY,influencedBYLabel
str,str,str,str
"""Q11708442""","""Halina Kalmanowicz""",null,null
"""Q3167907""","""Jean-Maurice Mourat""",null,null
"""Q5953683""","""Ali SamadPour""",null,null
"""Q29913453""","""Yella Pessl""",null,null
"""Q132076325""","""GON""",null,null
…,…,…,…
"""Q115010""","""Max von Thun""",null,null
"""Q7610784""","""Stephen Wade""",null,null
"""Q106567""","""Toby Gad""",null,null


In [75]:
#These are checks to ensure that size of the dataset has not mistakenly been reduced
print(f"The size of the dataset now is {len(artistDF["artist"].unique())}")

The size of the dataset now is 444888


In [76]:
#Look at selection of the musicBrainz table for any anomalies
mBtable = artistDF.select(pl.col(["artist","artistLabel","musicBrainzID"])).unique()
mB_issues = mBtable.filter(
    pl.col("musicBrainzID").str.contains("http")
).unique()
display(mB_issues)


mB_issues.write_csv("./missingValuesStore/artistsEntitiesWithFormattingIssuesInTheirMusicBrainzID.csv")
mB_users = mBtable.filter(
    pl.col("musicBrainzID").str.contains("user")
)
display(mB_users)


#Because it is two people who had these errored user links in there I will drop those two rows as their info will continue to be in the dataset
artistDF = artistDF.filter(
    (~pl.col("musicBrainzID").str.contains("user")) | (pl.col("musicBrainzID").is_null())
)

#Get the artist ID of the people with the links for theirs instead of the number
artistDF = artistDF.with_columns(
    musicBrainzID = pl.when(pl.col("musicBrainzID").str.contains("artist")).then(
        pl.col("musicBrainzID").str.split("/").list.last()
    ).otherwise(pl.col("musicBrainzID")),
)

#See a selection of artists to make sure changes were done correctly.
display(artistDF.filter((pl.col("artist") == "Q138387061") | (pl.col("artist") == "Q138413725") | (pl.col("artist") == "Q139919749")  ).unique())


artist,artistLabel,musicBrainzID
str,str,str
"""Q140085977""","""Confidence Survival Eyong""","""https://musicbrainz.org/artist…"
"""Q138589085""","""Cyprian Błażejczyk""","""https://musicbrainz.org/artist…"
"""Q140159484""","""Boyonset""","""https://musicbrainz.org/artist…"
"""Q139919749""","""Biko Manus""","""https://musicbrainz.org/artist…"
"""Q138413725""","""Konstantin Popov""","""https://musicbrainz.org/user/k…"
"""Q12216002""","""Saadoun Jaber""","""https://musicbrainz.org/artist…"
"""Q139623092""","""Mert Grey""","""https://musicbrainz.org/artist…"
"""Q138387061""","""Monster Fragment""","""https://musicbrainz.org/user/m…"


artist,artistLabel,musicBrainzID
str,str,str
"""Q138387061""","""Monster Fragment""","""https://musicbrainz.org/user/m…"
"""Q138413725""","""Konstantin Popov""","""https://musicbrainz.org/user/k…"


artist,occupation,gender,occupationLabel,genderLabel,countryCitizenship,influencedBY,influencedBYLabel,statementCount,genre,article,musicBrainzID,recordLabel,recordLabelLabel,countryCitizenshipLabel,genreLabel,artistLabel
str,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,str
"""Q138413725""","""Q639669""","""Q6581097""","""musician""","""male""","""Q219""",null,null,null,"""Q193207""",null,"""732910a2-5ee1-494b-9989-61afd6…",null,null,"""Bulgaria""","""ambient music""","""Konstantin Popov"""
"""Q138413725""","""Q183945""","""Q6581097""","""record producer""","""male""","""Q219""",null,null,null,"""Q9778""",null,"""732910a2-5ee1-494b-9989-61afd6…",null,null,"""Bulgaria""","""electronic music""","""Konstantin Popov"""
"""Q138413725""","""Q183945""","""Q6581097""","""record producer""","""male""","""Q219""",null,null,null,"""Q193207""",null,"""732910a2-5ee1-494b-9989-61afd6…",null,null,"""Bulgaria""","""ambient music""","""Konstantin Popov"""
"""Q138387061""","""Q639669""","""Q6581097""","""musician""","""male""","""Q142""",null,null,null,"""Q223840""",null,"""f681a0d9-b9bb-4555-9b22-ada68f…",null,null,"""France""","""hardstyle""","""Monster Fragment"""
"""Q139919749""","""Q183945""","""Q6581097""","""record producer""","""male""","""Q30""",null,null,null,null,null,"""0984057b-5ef5-4c92-80fa-585934…",null,null,"""United States""",null,"""Biko Manus"""
"""Q138413725""","""Q639669""","""Q6581097""","""musician""","""male""","""Q219""",null,null,null,"""Q9778""",null,"""732910a2-5ee1-494b-9989-61afd6…",null,null,"""Bulgaria""","""electronic music""","""Konstantin Popov"""


In [77]:
#These are checks to ensure that size of the dataset has not mistakenly been reduced
print(f"The size of the dataset now is {len(artistDF["artist"].unique())}")

The size of the dataset now is 444888


In [78]:
#Look at the data in the statements tab to ensure they are all numbers
print(artistDF.schema)

#By checking the schema I can see that the statementCount is rightfully a series of numbers (Float64)
#Did a mean to confirm that there were no strings in there that could cause an issue
artistDF.select(pl.col("statementCount")).unique().mean()

Schema({'artist': String, 'occupation': String, 'gender': String, 'occupationLabel': String, 'genderLabel': String, 'countryCitizenship': String, 'influencedBY': String, 'influencedBYLabel': String, 'statementCount': Float64, 'genre': String, 'article': String, 'musicBrainzID': String, 'recordLabel': String, 'recordLabelLabel': String, 'countryCitizenshipLabel': String, 'genreLabel': String, 'artistLabel': String})


statementCount
f64
333.764925


In [79]:
#Look at the wiki column and see that every article has the wiki link in there by checking for those without
artistDF.filter(
    (~pl.col("article").str.contains("https")) | (~pl.col("article").str.contains("wiki"))
).unique()

#From this I can see that all the articles in here are correct links by checking that they contain "https" and "wiki(pedia)*"

artist,occupation,gender,occupationLabel,genderLabel,countryCitizenship,influencedBY,influencedBYLabel,statementCount,genre,article,musicBrainzID,recordLabel,recordLabelLabel,countryCitizenshipLabel,genreLabel,artistLabel
str,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,str


In [80]:
artistDF = artistDF.unique()

At this point I have checked all the fields and attempted to clean up the data to highlight any unknowns. Each row should now be unique and contain a single value

In [89]:
#Look at the gender split of the data
#Discover the gender gap in the set and the number of people who have a null/unknown gender
artistsByGender = artistDF.select(
    pl.col("artist"),
    pl.col("gender"),
    pl.col("genderLabel"),
).unique()
artistGenderGrouped = artistsByGender.group_by(["gender", "genderLabel"]).agg(pl.col("artist").count())
artistGenderGrouped.write_csv("./Visuals Data/artistsGroupedByGender.csv", null_value="null/unknown")
artistGenderGroupedFRAC = artistsByGender["genderLabel"].value_counts(sort=True, normalize=True, name="fraction")
display(artistGenderGrouped)
display(artistGenderGroupedFRAC)

gender,genderLabel,artist
str,str,u32
"""Q12964198""","""genderqueer""",46
"""Q130477254""","""non-binary woman""",1
"""Q15145779""","""cisgender woman""",6
"""Q17148251""","""travesti""",7
"""Q138806923""","""intersex trans woman""",5
…,…,…
"""Q2449503""","""trans man""",107
"""Q1097630""","""intersex""",8
"""Q27679684""","""transfeminine""",15


genderLabel,fraction
str,f64
"""male""",0.745138
"""female""",0.251865
"""non-binary""",0.00106
"""trans woman""",0.001058
"""trans man""",0.00024
…,…
"""demiboy""",0.000002
"""apagender""",0.000002
"""gender agnostic""",0.000002


445127


In [90]:
#Look at the region/country split of the data
regions = artistDF.select(
    pl.col("artist"),
    pl.col("countryCitizenship"),
    pl.col("countryCitizenshipLabel"),
).unique()
regionsGrouped = regions.group_by(["countryCitizenship", "countryCitizenshipLabel"]).agg(pl.col("artist").count())
regionsGrouped.write_csv("./Visuals Data/artistsGroupedByRegions-Countries.csv", null_value="null/unknown")
display(regionsGrouped.sample(5))

countRegionsNull = artistDF.filter(
    pl.col("countryCitizenship").is_null()
).select(pl.col("artist")).unique()

countRegionsNotNull = artistDF.filter(
    pl.col("countryCitizenship").is_not_null()
).select(pl.col("artist")).unique()
print(f"There are {len(countRegionsNull)} artists without a region/country")
print(f"There are {len(countRegionsNotNull)} artists with a region/country")



countryCitizenship,countryCitizenshipLabel,artist
str,str,u32
"""Q137816""","""Taiwan under Japanese rule""",10
"""Q660702""","""Further Austria""",1
"""Q804""","""Panama""",122
"""Q320925""","""Shu Han""",1
"""Q347""","""Liechtenstein""",13


There are 122946 artists without a region/country
There are 321943 artists with a region/country


In [91]:
#Look at the genre and gender-genre split on wikidata
genresDF = artistDF.select(
    pl.col("artist"),
    pl.col("genderLabel"),
    pl.col("genre"),
    pl.col("genreLabel"),
).unique()
genresDFGrouped = genresDF.group_by(["genre", "genreLabel"]).agg(pl.col("artist").count())
genresDFGrouped.write_csv("./Visuals Data/artistsGroupedByGenres.csv", null_value="null/unknown")
display(genresDFGrouped.sample(5))
genreGenderSplit = genresDF.group_by(["genre", "genreLabel","genderLabel"]).agg(pl.col("artist").count())
print(genreGenderSplit.head())
genreGenderPivot = genreGenderSplit.pivot(
    "genderLabel",
    index = ["genre", "genreLabel"],
    values = "artist",
    aggregate_function = "sum"
)
genreGenderPivot.write_csv("./Visuals Data/artistsGroupedByGenresAndGender.csv", null_value="null/unknown")


countGenreNull = artistDF.filter(
    pl.col("genre").is_null()
).select(pl.col("artist")).unique()

countGenreNotNull = artistDF.filter(
    pl.col("genre").is_not_null()
).select(pl.col("artist")).unique()
print(f"There are {len(countGenreNull)} artists without a genre")
print(f"There are {len(countGenreNotNull)} artists with a genre")

genreGenderPivot


genre,genreLabel,artist
str,str,u32
"""Q5612172""","""grupera""",5
"""Q125455354""","""reggaelton""",1
"""Q109469638""","""indie dance""",1
"""Q2406633""","""broken beat""",2
"""Q1815364""","""makossa""",28


shape: (5, 4)
┌──────────┬────────────────────┬─────────────┬────────┐
│ genre    ┆ genreLabel         ┆ genderLabel ┆ artist │
│ ---      ┆ ---                ┆ ---         ┆ ---    │
│ str      ┆ str                ┆ str         ┆ u32    │
╞══════════╪════════════════════╪═════════════╪════════╡
│ Q2296495 ┆ garage punk        ┆ female      ┆ 1      │
│ Q11401   ┆ hip-hop            ┆ female      ┆ 614    │
│ Q1640319 ┆ experimental music ┆ male        ┆ 215    │
│ Q1426909 ┆ flamenco jazz      ┆ female      ┆ 1      │
│ Q11401   ┆ hip-hop            ┆ trans man   ┆ 2      │
└──────────┴────────────────────┴─────────────┴────────┘
There are 345319 artists without a region/country
There are 99569 artists with a region/country


genre,genreLabel,female,male,trans man,non-binary,trans woman,transfeminine,null,transgender,genderfluid,two-spirit,transmasculine,androgyne,cisgender woman,genderqueer,agender,undisclosed gender,travesti,bigender,gender agnostic,cisgender man,demiboy,intersex trans woman,intersex,intersex woman,neutral sex,demigirl,intersex man,takatāpui,non-binary woman,apagender
str,str,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
"""Q2296495""","""garage punk""",1,12,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""Q11401""","""hip-hop""",614,4333,2,17,3,1,0,0,4,0,0,0,0,1,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0
"""Q1640319""","""experimental music""",59,215,0,7,9,2,0,0,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""Q1426909""","""flamenco jazz""",1,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""Q9162904""","""avant-pop""",4,11,0,1,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""Q8031215""","""women's music""",1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""Q6799787""","""mbube""",0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""Q215911""","""serenade""",0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [163]:
#Look at the occupation and occupation/gender split
occupationDF = artistDF.select(
    pl.col("artist"),
    pl.col("genderLabel"),
    pl.col("occupation"),
    pl.col("occupationLabel"),
).unique()
occupationDFDFGrouped = occupationDF.group_by(["occupation", "occupationLabel"]).agg(pl.col("artist").count())
occupationDFDFGrouped.write_csv("./Visuals Data/artistsGroupedByOccupation.csv", null_value="null/unknown")
display(occupationDFDFGrouped.sample(5))
occupationGenderSplit = occupationDF.group_by(["occupation", "occupationLabel","genderLabel"]).agg(pl.col("artist").count())

occupationGenderPivot = occupationGenderSplit.pivot(
    "genderLabel",
    index = ["occupation", "occupationLabel"],
    values = "artist",
    aggregate_function = "sum"
)

occupationGenderPivot.write_csv("./Visuals Data/artistsGroupedByOccupationAndGender.csv", null_value="null/unknown")
display(occupationGenderPivot.sample(5))

occupation,occupationLabel,artist
str,str,u32
"""Q4087517""","""beatmaker""",52
"""Q66763670""","""country musician""",225
"""Q136489193""","""steel drummer""",2
"""Q104224753""","""dansband musician""",18
"""Q135936594""","""bass clarinetist""",3


occupation,occupationLabel,genderLabel,artist
str,str,str,u32
"""Q51397277""","""tocaor""","""female""",1
"""Q183945""","""record producer""","""transmasculine""",2
"""Q584301""","""bassist""","""trans man""",1
"""Q60723829""","""pop singer""","""trans woman""",3
"""Q60723829""","""pop singer""","""genderqueer""",1


Old Code that I will ignore for now

In [84]:
cleanedArtistDF = artistDF.filter(
    ~pl.col("artist").is_in(pl.col("artistLabel").implode()),
    ~pl.col("influencedBY").is_in(pl.col("influencedBYLabel").implode()),
    ~pl.col("recordLabel").is_in(pl.col("recordLabelLabel").implode()),
    ~pl.col("genre").is_in(pl.col("genreLabel").implode()),
)

#Add a column for presence of wikiPage
cleanedArtistDF = cleanedArtistDF.with_columns(
    isThereAWiki = pl.col("article").is_not_null()
)
#Finally de-duplicate the data.
cleanedArtistDF = cleanedArtistDF.unique()
display(cleanedArtistDF.head(5))
display(cleanedArtistDF.shape)

artist,occupation,gender,occupationLabel,genderLabel,countryCitizenship,influencedBY,influencedBYLabel,statementCount,genre,article,musicBrainzID,recordLabel,recordLabelLabel,countryCitizenshipLabel,genreLabel,artistLabel,isThereAWiki
str,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str,str,bool
"""Q194220""","""Q36834""","""Q6581097""","""composer""","""male""","""Q30""","""Q364214""","""Rakim""",null,"""Q753679""","""https://en.wikipedia.org/wiki/…","""cfbc0924-0035-4d6c-8197-f02465…","""Q183387""","""Columbia Records""","""United States""","""gangsta rap""","""Nas""",true
"""Q10681""","""Q36834""","""Q6581097""","""composer""","""male""","""Q142""","""Q3176402""","""Jef Le Penven""",222.0,"""Q910606""","""https://en.wikipedia.org/wiki/…","""abc61045-7ae3-48d9-91f2-d44689…","""Q21077""","""Warner Music Group""","""France""","""music of Brittany""","""Alan Stivell""",true
"""Q15123969""","""Q488205""","""Q48270""","""singer-songwriter""","""non-binary""","""Q145""","""Q182725""","""Etta James""",192.0,"""Q37073""","""https://en.wikipedia.org/wiki/…","""5a85c140-dcf9-4dd2-b2c8-aff047…","""Q193023""","""Capitol Records""","""United Kingdom""","""pop music""","""Sam Smith""",true
"""Q44479972""","""Q753110""","""Q6581097""","""songwriter""","""male""","""Q30""","""Q6107""","""Tupac Shakur""",null,"""Q15697534""","""https://en.wikipedia.org/wiki/…","""2282c539-05a2-477c-82b7-fe4079…","""Q59210322""","""Scumgang Records""","""United States""","""drill""","""6ix9ine""",true
"""Q230454""","""Q639669""","""Q6581072""","""musician""","""female""","""Q16""","""Q272913""","""Kitty Wells""",256.0,"""Q37073""","""https://en.wikipedia.org/wiki/…","""bf4bb21e-759f-4b22-bfd4-174593…","""Q843402""","""Warner Bros. Records""","""Canada""","""pop music""","""k.d. lang""",true


(136735, 18)

That is the end of the cleaning stage. Now time to analyse the data. Null values were left intentionally as we are looking for the gaps

In [85]:
#Discover the gender gap in the set.
artistsByGender = cleanedArtistDF.select(
    pl.col("artist"),
    pl.col("gender"),
    pl.col("genderLabel"),
)
artistsByGender = artistsByGender.unique()
artistGenderGrouped = artistsByGender.group_by(["gender", "genderLabel"]).agg(pl.col("artist").count())
artistGenderGroupedFRAC = artistsByGender["genderLabel"].value_counts(sort=True, normalize=True, name="fraction")
display(artistGenderGrouped)
display(artistGenderGroupedFRAC)

gender,genderLabel,artist
str,str,u32
"""Q6581097""","""male""",303
"""Q18116794""","""genderfluid""",4
"""Q1052281""","""trans woman""",1
"""Q6581072""","""female""",188
"""Q48270""","""non-binary""",4


genderLabel,fraction
str,f64
"""male""",0.606
"""female""",0.376
"""non-binary""",0.008
"""genderfluid""",0.008
"""trans woman""",0.002


In [86]:
genderChart = (
    alt.Chart(artistGenderGrouped).mark_bar().encode(
    y="artist",
    x="genderLabel",
    )
    .properties(width=500, title="Gender Distribution of Wikidata")
)
genderChart.encoding.x.title = "Genders"
genderChart.encoding.y.title = "Counts"
genderChart.show()



genderBase = alt.Chart(artistGenderGrouped).encode(
    alt.Theta("artist:Q").stack(True),
    alt.Color("genderLabel:N").legend(None)
    )
pie = genderBase.mark_arc(outerRadius=120)
text = genderBase.mark_text(radius=140, size=20).encode(text="genderLabel:N")
pie+text

alt.Chart(...)

alt.LayerChart(...)

In [88]:
#Top genres and genre gap

In [89]:
#Top occupations

In [90]:
#Top occupations by gender

In [91]:
#Gender and genre split